In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib seaborn

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    ASTFeatureExtractor,
    ASTForAudioClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from datasets import load_dataset, Audio, DatasetDict

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Φόρτωση συνόλου δεδομένων και speaker independent διαχωρισμός

In [ ]:
def load_tess_speaker_independent():
    """
    Συνάρτηση υπεύθυνη για τον speaker independent διαχωρισμό. Η ομιλήτρια OAF
    αντιστοιχίζεται στο σύνολο εκπαίδευσης και η YAF στο σύνολο δοκιμής
    """
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF", load_from_cache_file=False)
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF", load_from_cache_file=False)

    #Μετατροπή συχνότητας δειγματοληψίας σε 16kHz
    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()
print(dataset)

In [ ]:
#Εξαγωγή των ετικετών των συναισθημάτων σε λίστα
labels = sorted(set(dataset["train"]["emotion"]))

#Δημιουργία λεξικών αντιστοίχισης ετικετών και αριθμών
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

#Υπολογισμός αριθμού ετικετών
num_labels = len(labels)
print(f"Διαθέσιμες ετικέτες: {labels}")
print(f"Αριθμός ετικετών: {num_labels}")

In [ ]:
"""
Λόγω του μεγέθους της εισόδου του AST (1024 x 128 ανά δείγμα), η προεπεξεργασία δεν
εφαρμόζεται εκ των προτέρων σε όλο το dataset μέσω map, καθώς αυτό υπερβαίνει τη
διαθέσιμη μνήμη. Αντίθετα, η μετατροπή σε log-mel filterbank γίνεται δυναμικά ανά batch,
μέσα στον Data Collator. Το dataset διατηρεί μόνο το ακατέργαστο σήμα και την ετικέτα.
"""

keep_cols = ["audio", "emotion"]
dataset = dataset.remove_columns(
    [c for c in dataset["train"].column_names if c not in keep_cols]
)
print(dataset)

### Φόρτωση του αρχικού προεκπαιδευμένου μοντέλου

In [ ]:
model_repo = "MIT/ast-finetuned-audioset-10-10-0.4593"

#Φόρτωση του feature extractor του βασικού μοντέλου (log-mel filterbank 128 καναλιών, σταθερό μήκος 1024 frames)
feature_extractor = ASTFeatureExtractor.from_pretrained(model_repo)

#Φόρτωση του προεκπαιδευμένου μοντέλου με νέα κεφαλή ταξινόμησης
#(η κεφαλή των 527 κλάσεων του AudioSet αντικαθίσταται, εξ ου και το ignore_mismatched_sizes)
model = ASTForAudioClassification.from_pretrained(
    model_repo,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,
)

#Φόρτωση μοντέλου στην GPU
model.to("cuda")
print(f"Το μοντέλο φορτώθηκε στη συσκευή: {model.device}")

### Data Collator

In [ ]:
MAX_LENGTH = 6 * 16000  # 6 δευτερόλεπτα στα 16kHz

#Data Collator: περικοπή στα 6s, μετατροπή σε log-mel filterbank και μετατροπή ετικετών, ανά batch
class DataCollatorASTOnTheFly:
    def __init__(self, feature_extractor, label2id, max_length):
        self.feature_extractor = feature_extractor
        self.label2id = label2id
        self.max_length = max_length

    def __call__(self, features):
        #Περικοπή στα 6 δευτερόλεπτα πριν τον feature extractor (αυτός συμπληρώνει πάντα στα 1024 frames)
        audio_arrays = [f["audio"]["array"][:self.max_length] for f in features]

        #Μετατροπή σε κανονικοποιημένο log-mel filterbank, σχήμα (batch, 1024, 128)
        inputs = self.feature_extractor(
            audio_arrays,
            sampling_rate=16000,
            return_tensors="pt",
        )

        #Μετατροπή της ετικέτας από string σε integer id, για συμβατότητα με τον trainer
        labels = torch.tensor(
            [self.label2id[f["emotion"]] for f in features], dtype=torch.long
        )
        return {"input_values": inputs["input_values"], "labels": labels}

data_collator = DataCollatorASTOnTheFly(feature_extractor, label2id, MAX_LENGTH)

### Ορισμός μετρικών αξιολόγησης

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
from scipy.special import softmax

In [ ]:
#Λίστα για την αποθήκευση των confusion matrices
all_confusion_matrices = []

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)
    labels_bin = label_binarize(labels_true, classes=list(range(num_labels)))

    # ROC-AUC (macro-averaged, one-vs-rest)
    try:
        roc_auc = roc_auc_score(labels_bin, probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = float("nan")

    # PR-AUC (macro-averaged average precision)
    try:
        pr_auc = average_precision_score(labels_bin, probs, average="macro")
    except ValueError:
        pr_auc = float("nan")

    #Σταθερό σχήμα πίνακα ακόμα κι αν κάποια κλάση δεν προβλεφθεί καθόλου
    cm = confusion_matrix(labels_true, predictions, labels=list(range(num_labels)))
    all_confusion_matrices.append(cm)

    return {
        "accuracy": accuracy_score(labels_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels_true, predictions),
        "precision": precision_score(labels_true, predictions, average="weighted", zero_division=0),
        "recall": recall_score(labels_true, predictions, average="weighted", zero_division=0),
        "f1": f1_score(labels_true, predictions, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(labels_true, predictions),
        "k_score": cohen_kappa_score(labels_true, predictions),
        "log_loss": log_loss(labels_true, probs, labels=list(range(num_labels))),
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
    }

### Ορισμός παραμέτρων εκπαίδευσης

In [ ]:
#Δυναμικός υπολογισμός warmup_steps (10% των συνολικών training steps)
per_device_train_batch_size = 8
gradient_accumulation_steps = 4
num_train_epochs = 8

effective_batch_size = per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = len(dataset["train"]) // effective_batch_size
total_steps = steps_per_epoch * num_train_epochs
warmup_steps = max(1, int(0.1 * total_steps))
print(f"Steps/epoch: {steps_per_epoch}, Total steps: {total_steps}, Warmup steps: {warmup_steps}")

#Καθορισμός παραμέτρων εκπαίδευσης του μοντέλου
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/SER_Thesis/ast_tess_checkpoints",
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=gradient_accumulation_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=num_train_epochs,
    fp16=False,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

In [ ]:
#Καθαρισμός των confusion matrix, από τυχόν προηγούμενες εκτελέσεις
all_confusion_matrices.clear()

### Δημιουργία και Εκπαίδευση του Trainer

In [ ]:
#Κλήση του trainer και εισαγωγή των παραμέτρων εκπαίδευσης
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Εκκίνηση εκπαίδευσης...")
trainer.train()
print("Εκπαίδευση ολοκληρώθηκε!")

In [ ]:
#Εκτύπωση των τελικών αποτελεσμάτων της εκπαίδευσης
eval_results = trainer.evaluate()
print(f"Αποτελέσματα αξιολόγησης: {eval_results}")

### Οπτικοποίηση αποτελεσμάτων (Confusion Matrices)

In [ ]:
#Αποθήκευση των ετικετών για τους άξονες των matrices
labels = [
    model.config.id2label[i]
    for i in sorted(model.config.id2label.keys())
]

#Loop στην λίστα με τα confusion matrices
for epoch, cm in enumerate(all_confusion_matrices):
  #Δημιουργία γραφήματος
  plt.figure(figsize=(6,5))

  #Δημιουργία Heatmap
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      xticklabels=labels,
      yticklabels=labels
  )

  plt.title(f"Confusion Matrix - Epoch {epoch+1}")
  plt.xlabel("Predicted")
  plt.ylabel("True")
  plt.show()

In [ ]:
### Αποθήκευση εκπαιδευμένου μοντέλου στο HuggingFace Hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
repo_id = "VasilisAsim/ast-finetuned-TESS"

#Αποθήκευση του μοντέλου στο HuggingFace
model.push_to_hub(repo_id)

#Αποθήκευση του Feature Extractor
feature_extractor.push_to_hub(repo_id)

print(f"Το μοντέλο και ο επεξεργαστής αποθηκεύτηκαν στο Hugging Face Hub με ID: {repo_id}")

In [ ]:
from huggingface_hub import repo_info
info = repo_info("VasilisAsim/ast-finetuned-TESS")
print(info.last_modified, info.sha)